In [ ]:
import glob
import os

import h5py
import keras
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.optimizers import Adam
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix, fbeta_score, )
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    Input, Flatten, Dense, Conv2D, Dropout
)

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

best_params = None


In [ ]:
data_folder = '/Users/logan/brain_strain_cnn/rugby_brain_strain_CNN/data/ellesmere_u16'

# Make sure the path exists
if not os.path.exists(data_folder):
    print("Folder does not exist:", data_folder)
else:
    # Recursive search for all .h5 files
    file_list = glob.glob(os.path.join(data_folder, '**', '*.h5'), recursive=True)
    print(f"Found {len(file_list)} files:")
    for f in file_list:
        print(f)

X = []
y = []

for file_path in file_list:
    with h5py.File(file_path, 'r') as hf:
        for key in hf.keys():
            group = hf[key]
            pred = group.attrs.get('QA').astype(bool)

            dset_names = list(group.keys())
            if pred is False or pred == 0:
                perm_names = [n for n in dset_names if n.startswith('perm_') and not n.startswith('lin_perm_')]
            else:
                perm_names = ['perm_xyz'] if 'perm_xyz' in dset_names else []

            for perm_name in perm_names:
                rot = group[perm_name][:]

                if rot.ndim != 3 or rot.shape[0] != 1:
                    continue

                # Convert (1, 3, L) -> (3, L, 1)
                rot = rot.transpose(1, 2, 0)

                X.append(rot)
                y.append(pred)

# Convert to numpy arrays
if len(X) > 0:
    X = np.stack(X, axis=0)
    y = np.array(y)
else:
    X = np.array([])
    y = np.array([])

# 80:20 train-test split
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
        stratify=y
    )

    input_shape = X_train.shape[1:]

    print(f"Total samples: {len(X)}")
    print(f"Train samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")

else:
    X_train, X_test = [], []
    y_train, y_test = [], []
    input_shape = (3, 1000, 1)



In [ ]:
# Alternative loader (stack rot+lin for all samples)
# QA True: use perm_xyz + lin_perm_xyz stacked as channels -> (3, L, 2)
# QA False: use all perm_* paired with matching lin_perm_* stacked as channels

data_folder = '/Users/logan/brain_strain_cnn/rugby_brain_strain_CNN/data/ellesmere_u16'

if not os.path.exists(data_folder):
    print("Folder does not exist:", data_folder)
else:
    file_list = glob.glob(os.path.join(data_folder, '**', '*.h5'), recursive=True)
    print(f"Found {len(file_list)} files:")
    for f in file_list:
        print(f)

X = []
y = []

for file_path in file_list:
    with h5py.File(file_path, 'r') as hf:
        for key in hf.keys():
            group = hf[key]
            pred = group.attrs.get('QA').astype(bool)

            if pred is False or pred == 0:
                rot_names = [n for n in group.keys() if n.startswith('perm_') and not n.startswith('lin_perm_')]
            else:
                rot_names = ['perm_xyz'] if 'perm_xyz' in group else []

            for rot_name in rot_names:
                lin_name = rot_name.replace('perm_', 'lin_perm_', 1)
                if lin_name not in group:
                    continue

                rot = group[rot_name][:]
                lin = group[lin_name][:]
                if rot.ndim != 3 or lin.ndim != 3 or rot.shape != lin.shape or rot.shape[0] != 1:
                    continue

                rot = rot.transpose(1, 2, 0)
                lin = lin.transpose(1, 2, 0)
                sample = np.concatenate([rot, lin], axis=2)  # (3, L, 2)

                X.append(sample)
                y.append(pred)

if len(X) > 0:
    X = np.stack(X, axis=0)
    y = np.array(y)
else:
    X = np.array([])
    y = np.array([])

if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
        stratify=y
    )

    input_shape = X_train.shape[1:]

    print(f"Total samples: {len(X)}")
    print(f"Train samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")

else:
    X_train, X_test = [], []
    y_train, y_test = [], []
    input_shape = (3, 1000, 2)



In [ ]:
def build_model(params, input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel1"],
            strides=params["stride1"],
            activation='relu',
            padding='valid'
        ),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel2"],
            strides=params["stride2"],
            activation='relu',
            padding='valid'
        ),
        Conv2D(
            filters=params["filters"],
            kernel_size=params["kernel3"],
            strides=params["stride3"],
            activation='relu',
            padding='valid'
        ),
        Dropout(params.get("dropout", 0.2)),
        Flatten(),
        Dense(units=params["dense_units"], activation='relu'),
        Dense(units=1, activation='sigmoid')
    ])

    optimizer = Adam(learning_rate=params.get("lr", 1e-4))
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
            tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
        ]
    )
    return model

In [ ]:
param_grid = [
    {
        "filters": 16,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 10,
    },
    {
        "filters": 16,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 32,
    },
    {
        "filters": 32,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 10,
    },
    {
        "filters": 32,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 32,
    },
    {
        "filters": 32,
        "kernel1": (3, 7),
        "stride1": (1, 2),
        "kernel2": (1, 7),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 10,
    },
    {
        "filters": 32,
        "kernel1": (3, 7),
        "stride1": (1, 2),
        "kernel2": (1, 7),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 32,
    },
]

if len(X_train) > 0:
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_results = []
    thresholds = np.linspace(0.05, 0.95, 19)

    for params in param_grid:
        fold_scores = []
        fold_thresholds = []
        for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            model = build_model(params, input_shape=X_train.shape[1:])

            early_stopping = EarlyStopping(
                monitor='val_loss',
                patience=8,
                mode='min',
                restore_best_weights=True
            )

            model.fit(
                X_tr,
                y_tr,
                epochs=60,
                batch_size=64,
                validation_data=(X_val, y_val),
                callbacks=[early_stopping],
                verbose=0
            )

            y_prob = model.predict(X_val, verbose=0).ravel()
            best_f2 = -1.0
            best_t = 0.5
            for t in thresholds:
                y_pred = (y_prob >= t).astype(int)
                f2 = fbeta_score(y_val, y_pred, beta=2)
                if f2 > best_f2:
                    best_f2 = f2
                    best_t = t
            fold_scores.append(best_f2)
            fold_thresholds.append(best_t)

        mean_f2 = float(np.mean(fold_scores))
        mean_t = float(np.mean(fold_thresholds))
        cv_results.append((mean_f2, mean_t, params))
        print(f"Params {params} -> mean F2: {mean_f2:.4f} @ mean thr {mean_t:.3f}")

    best_f2, best_t, best_params = max(cv_results, key=lambda x: x[0])
    print(f"Best params: {best_params}")
    print(f"Best mean F2: {best_f2:.4f} @ mean thr {best_t:.3f}")
else:
    best_params = None


In [ ]:
input_shape = X_train.shape[1:] if len(X_train) else input_shape

if "build_model" not in globals():
    def build_model(params, input_shape):
        model = Sequential([
            Input(shape=input_shape),
            Conv2D(
                filters=params["filters"],
                kernel_size=params["kernel1"],
                strides=params["stride1"],
                activation='relu',
                padding='valid'
            ),
            Conv2D(
                filters=params["filters"],
                kernel_size=params["kernel2"],
                strides=params["stride2"],
                activation='relu',
                padding='valid'
            ),
            Conv2D(
                filters=params["filters"],
                kernel_size=params["kernel3"],
                strides=params["stride3"],
                activation='relu',
                padding='valid'
            ),
            Dropout(params.get("dropout", 0.2)),
            Flatten(),
            Dense(units=32, activation='relu'),
            Dense(units=1, activation='sigmoid')
        ])

        optimizer = Adam(learning_rate=params.get("lr", 1e-4))
        model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=[
                keras.metrics.AUC(curve="PR", name="pr_auc"),
                keras.metrics.AUC(curve="ROC", name="roc_auc"),
            ]
        )
        return model

default_params = {
    "filters": 32,
    "kernel1": (3, 10),
    "stride1": (1, 2),
    "kernel2": (1, 10),
    "stride2": (1, 2),
    "kernel3": (1, 5),
    "stride3": (1, 1),
    "dense_units": 32,

}

if "best_params" in globals() and best_params:
    model = build_model(best_params, input_shape)
else:
    model = build_model(default_params, input_shape)

model.summary()



In [ ]:
early_stopping = EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=20,
    restore_best_weights=True
)

# Model training
history = model.fit(
    X_train,
    y_train,
    epochs=250,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    shuffle=True
)



In [ ]:
model.save('cnn_location.keras')

In [ ]:
loaded_model = keras.saving.load_model("brain_strain_cnn.keras")


In [ ]:
# Evaluation metrics and confusion matrix
if len(X_test) > 0:
    y_prob = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob >= 0.15).astype(int)

    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("PR-AUC:", average_precision_score(y_test, y_prob))
    print(classification_report(y_test, y_pred, digits=4))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

